# régler xgboost (hyperparameter tuning)

on essaie plusieurs réglages au hasard et on garde le meilleur.

**important** : le jeu de test ne sert pas à choisir. on coupe l'entraînement en deux : une partie pour apprendre, une partie **validation** pour comparer les réglages. le test ne sert qu'à la note finale, une seule fois.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

train = pd.read_parquet("../data/train/train.parquet")
test = pd.read_parquet("../data/test/test.parquet")

classes = list("ABCDEFG")
vers_chiffre = {c: i for i, c in enumerate(classes)}
nombres = ["surface", "niveaux", "hauteur", "apres_2026"]
textes = ["periode", "type_batiment", "energie", "installation", "equipement",
          "energie_eau_chaude", "departement", "altitude"]
colonnes = nombres + textes


def notes(vrai, predit):
    return {
        "exactitude": round(accuracy_score(vrai, predit), 3),
        "à une classe près": round(np.mean(np.abs(predit - vrai) <= 1), 3),
        "f1 macro": round(f1_score(vrai, predit, average="macro"), 3),
        "passoire précision": round(precision_score(vrai >= 5, predit >= 5), 3),
        "passoire rappel": round(recall_score(vrai >= 5, predit >= 5), 3),
    }


def creer_modele(**reglages):
    return Pipeline([
        ("prep", ColumnTransformer(
            [("texte", OneHotEncoder(handle_unknown="ignore"), textes)],
            remainder="passthrough",
        )),
        ("xgb", XGBClassifier(n_jobs=4, **reglages)),
    ])


def poids_adoucis(y):
    return np.sqrt(compute_sample_weight("balanced", y))

## 1. couper l'entraînement : apprentissage + validation

pour aller plus vite, on règle sur 100 000 logements (la courbe d'apprentissage montrait que ça suffit presque).

In [2]:
part = train.sample(100_000, random_state=42)
appr, valid = train_test_split(part, test_size=0.2, stratify=part["classe"], random_state=42)
y_appr = appr["classe"].map(vers_chiffre).to_numpy()
y_valid = valid["classe"].map(vers_chiffre).to_numpy()

## 2. essayer 12 réglages au hasard

- **n_estimators** : nombre d'arbres
- **max_depth** : nombre de questions par arbre (profondeur)
- **learning_rate** : à quel point chaque arbre corrige le précédent
- **min_child_weight** : nombre minimum de logements au bout d'une branche (évite d'apprendre par cœur)
- **subsample** / **colsample_bytree** : part des logements / des colonnes que voit chaque arbre

on compare avec le **f1 macro**, qui tient compte des classes rares.

In [3]:
hasard = np.random.default_rng(42)
grille = {
    "n_estimators": [200, 400, 700],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 5, 20],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}
# le réglage de départ, pour comparer
essais = [{"n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
           "min_child_weight": 1, "subsample": 1.0, "colsample_bytree": 1.0}]
essais += [{k: v[hasard.integers(len(v))] for k, v in grille.items()} for _ in range(12)]

tableau = []
for i, reglages in enumerate(essais):
    m = creer_modele(**reglages)
    m.fit(appr[colonnes], y_appr, xgb__sample_weight=poids_adoucis(y_appr))
    tableau.append({**reglages, **notes(y_valid, m.predict(valid[colonnes]))})
    print(f"essai {i} : f1 macro = {tableau[-1]['f1 macro']}")

tableau = pd.DataFrame(tableau).sort_values("f1 macro", ascending=False)
tableau

essai 0 : f1 macro = 0.398


essai 1 : f1 macro = 0.4


essai 2 : f1 macro = 0.39


essai 3 : f1 macro = 0.393


essai 4 : f1 macro = 0.399


essai 5 : f1 macro = 0.403


... (4 outputs left out)

essai 10 : f1 macro = 0.385


essai 11 : f1 macro = 0.403


essai 12 : f1 macro = 0.396


,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,exactitude,à une classe près,f1 macro,passoire précision,passoire rappel
5,700,8,0.05,20,0.85,0.8,0.468,0.860,0.403,0.429,0.437
11,700,8,0.05,20,0.85,0.6,0.468,0.861,0.403,0.431,0.431
7,700,10,0.03,5,0.70,1.0,0.467,0.858,0.400,0.422,0.428
1,200,10,0.05,5,0.85,1.0,0.470,0.863,0.400,0.440,0.419
4,700,6,0.05,5,0.70,1.0,0.465,0.859,0.399,0.426,0.426
0,300,6,0.10,1,1.00,1.0,0.464,0.862,0.398,0.439,0.406
8,700,6,0.03,20,0.85,1.0,0.464,0.863,0.396,0.440,0.410
12,700,6,0.03,5,1.00,0.6,0.464,0.863,0.396,0.453,0.392
3,700,10,0.10,20,0.85,0.6,0.460,0.850,0.393,0.394,0.435
9,700,10,0.10,1,0.85,0.8,0.462,0.849,0.393,0.390,0.397


## 3. note finale : meilleur réglage, entraîné sur tout l'entraînement, noté sur le test

In [4]:
meilleur = essais[tableau.index[0]]  # le réglage avec le meilleur f1 macro
print("meilleur réglage :", meilleur)

y_train = train["classe"].map(vers_chiffre).to_numpy()
y_test = test["classe"].map(vers_chiffre).to_numpy()

final = {}
for nom, reglages in [("réglage de départ", essais[0]), ("meilleur réglage", meilleur)]:
    m = creer_modele(**reglages)
    m.fit(train[colonnes], y_train, xgb__sample_weight=poids_adoucis(y_train))
    final[nom] = notes(y_test, m.predict(test[colonnes]))
pd.DataFrame(final)

meilleur réglage : {'n_estimators': 700, 'max_depth': 8, 'learning_rate': 0.05, 'min_child_weight': 20, 'subsample': 0.85, 'colsample_bytree': 0.8}


,réglage de départ,meilleur réglage
exactitude,0.475,0.479
à une classe près,0.865,0.865
f1 macro,0.409,0.416
passoire précision,0.456,0.449
passoire rappel,0.414,0.436
